# 02 — Baseline & Experiments Evaluation

**Project**: Black-and-White Image Colorization with Deep Learning

This notebook shows the results of the experiments trained with `python -m src.train`.
It displays the full 12,000-image benchmark once the Baseline, E1, and E2 runs are available, and the
smaller 1,200-image pilot until then.

| Run | Model | What changes |
| --- | --- | --- |
| Baseline | Simple CNN Autoencoder (MSE, Adam, `lr=2e-4`, `bs=16`, 30 epochs) | reference |
| E1 | Simple CNN Autoencoder | learning rate `2e-4 → 1e-3` |
| E2 | U-Net with ImageNet-pretrained ResNet-18 encoder (L1 loss) | Method 1 |
| M3 | Palette conditional diffusion (L2 noise loss, DDIM sampling) | Method 3 |
| E6 | Palette | DDIM sampling steps 100 and 250 (Scenario 6) |

### Setup

Notebooks live in `notebooks/`, but the code imports the `src` package from the project root.
This cell finds the root (one level up when started from `notebooks/`) and adds it to `sys.path`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.models import build_model
from src.utils import count_parameters

### Model sizes

Builds each architecture (without downloading pretrained weights) and counts its parameters.
Similar sizes make the comparison fair: a better score should come from the design, not from a much
bigger network.

In [ ]:
architectures = [
    ('SimpleCNNAutoencoder (Baseline)', {'name': 'baseline_ae', 'base_channels': 64}),
    ('UNetResNet18 (Method 1)', {'name': 'unet_resnet18', 'pretrained': False}),
    ('Palette diffusion U-Net (Method 3)', {'name': 'palette'}),
]
for label, model_cfg in architectures:
    total, trainable = count_parameters(build_model(model_cfg))
    print(f'{label:36s} | Total params: {total:>12,} | Trainable: {trainable:>12,}')

## 1. Test-set results

`results/metrics_summary.csv` has one row per run, written after the best checkpoint (lowest validation
loss) is evaluated on the 1,200 test images.

- **PSNR** ↑ and **SSIM** ↑ reward predictions close to the original colors.
- **LPIPS** ↓ measures perceptual similarity and is more forgiving of a different but plausible color.
- **FID** ↓ compares the distribution of all 1,200 colorized test images with the real ones (Inception-v3
  features). It does not compare image pairs, so it rewards realistic, saturated colors even when they
  differ from the original, which is where faded regression outputs score worst. It is empty for runs
  made before FID was added; `python -m src.train --config <config> --eval-only` fills it in.
- `sampling_steps` is filled only for Palette runs; `test_time_sec` shows how long inference took.

In [ ]:
full_results = ROOT / 'results'
full_summary = full_results / 'metrics_summary.csv'
expected = {'baseline_cnn_ae', 'e1_baseline_lr_1e3', 'e2_unet_resnet18'}
full_complete = (
    full_summary.exists()
    and expected <= set(pd.read_csv(full_summary)['experiment_name'])
    and (full_results / 'plots' / 'train_val_loss_curves.png').exists()
)
RESULTS = full_results if full_complete else full_results / 'pilot' / 'results'
print('Full 12,000-image benchmark' if full_complete else '1,200-image pilot')

summary_df = pd.read_csv(RESULTS / 'metrics_summary.csv')
columns = [
    'experiment_name', 'model', 'loss_fn', 'optimizer', 'lr', 'batch_size', 'epochs', 'sampling_steps',
    'best_epoch', 'total_params', 'train_time_min', 'test_time_sec', 'test_psnr', 'test_ssim', 'test_lpips',
    'test_fid',
]
display(summary_df[[c for c in columns if c in summary_df.columns]])

## 2. Training and validation curves

Generated by `python -m src.evaluate`.
- **Loss curves**: validation loss rising while training loss keeps falling means overfitting;
  both staying high means underfitting.
- **Metric curves**: PSNR, SSIM, and LPIPS on the validation set (solid) and on a fixed sample of 256
  training images (dashed), measured on epochs 1, 5, 10, … and the last epoch.

In [ ]:
for title, rel_path in [
    ('Train vs. Validation Loss Curves', 'plots/train_val_loss_curves.png'),
    ('Training and Validation PSNR, SSIM, and LPIPS', 'plots/val_metrics_comparison.png'),
]:
    plot_path = RESULTS / rel_path
    if not plot_path.exists():
        print('Missing plot:', plot_path)
        continue
    plt.figure(figsize=(15, 5))
    plt.imshow(Image.open(plot_path))
    plt.axis('off')
    plt.title(title)
    plt.show()

## 3. Qualitative comparison

The same test images colorized by every model, between the grayscale input and the ground truth.
Look for faded colors (typical for MSE regression), color bleeding across object edges, and whether
skies, skin, and vegetation get plausible colors.

In [ ]:
grid_path = RESULTS / 'samples' / 'comparison_grid.png'
if grid_path.exists():
    plt.figure(figsize=(16, 18))
    plt.imshow(Image.open(grid_path))
    plt.axis('off')
    plt.show()
else:
    print('Missing grid:', grid_path)